# 🧹 Notebook 1 — Data Cleaning

**Goal:** Transform raw `online_retail_II.csv` into a clean, analysis-ready dataset.

**Dataset:** UCI Online Retail II (2009–2011)  
**Raw rows:** ~1,067,371 | **Clean rows:** ~779,425

---
### Cleaning Steps
1. Load raw CSV  
2. Drop rows missing `Customer ID`  
3. Remove duplicate rows  
4. Remove returns (Quantity ≤ 0)  
5. Remove invalid prices (Price ≤ 0)  
6. Engineer `Revenue = Quantity × Price`  
7. Parse `InvoiceDate` to datetime  
8. Add `MonthYear` period column  
9. Export `cleaned_ecommerce.csv`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded ✅')

## 1️⃣ Load Raw Data

In [ ]:
df = pd.read_csv('../online_retail_II.csv', encoding='ISO-8859-1')
print(f'Raw shape: {df.shape}')
df.head()

In [ ]:
print('=== Data Types ===')
print(df.dtypes)
print('\n=== Missing Values ===')
print(df.isnull().sum())
print(f'\nMissing % for Customer ID: {df["Customer ID"].isnull().mean()*100:.1f}%')

In [ ]:
df.describe()

## 2️⃣ Drop Missing Customer IDs

In [ ]:
before = len(df)
df = df.dropna(subset=['Customer ID'])
after = len(df)
print(f'Dropped {before - after:,} rows with missing Customer ID')
print(f'Remaining: {after:,} rows')

## 3️⃣ Remove Duplicates

In [ ]:
before = len(df)
df = df.drop_duplicates()
print(f'Removed {before - len(df):,} duplicate rows')

## 4️⃣ Filter Invalid Transactions

In [ ]:
# Negative quantity = returns/cancellations
print(f'Rows with Quantity <= 0: {(df["Quantity"] <= 0).sum():,}')
df = df[df['Quantity'] > 0]

# Price must be positive
print(f'Rows with Price <= 0: {(df["Price"] <= 0).sum():,}')
df = df[df['Price'] > 0]

print(f'\nClean shape: {df.shape}')

## 5️⃣ Feature Engineering

In [ ]:
# Revenue per line item
df['Revenue'] = df['Quantity'] * df['Price']

# Parse dates
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

# Month-Year period for time series
df['MonthYear'] = df['InvoiceDate'].dt.to_period('M')

print('New columns added:')
print(df[['Revenue', 'InvoiceDate', 'MonthYear']].head())

## 6️⃣ Quick Visualisation — Data Quality Check

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(df['Quantity'].clip(0, 100), bins=50, color='#2563EB', edgecolor='white')
axes[0].set_title('Quantity Distribution (clipped at 100)')
axes[0].set_xlabel('Quantity')

axes[1].hist(df['Price'].clip(0, 50), bins=50, color='#F59E0B', edgecolor='white')
axes[1].set_title('Price Distribution (clipped at £50)')
axes[1].set_xlabel('Price (£)')

axes[2].hist(df['Revenue'].clip(0, 500), bins=50, color='#10B981', edgecolor='white')
axes[2].set_title('Revenue per Line (clipped at £500)')
axes[2].set_xlabel('Revenue (£)')

plt.suptitle('Distribution Checks After Cleaning', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 7️⃣ Export Clean Dataset

In [ ]:
df.to_csv('cleaned_ecommerce.csv', index=False)
print(f'✅ Exported cleaned_ecommerce.csv — {len(df):,} rows, {df.shape[1]} columns')
print(df.columns.tolist())